<!-- type: how-to -->
# Lag-Aware ModMRMR → ForecastPrepContract

**Goal:** show the minimal end-to-end path from raw time series data to a
`ForecastPrepContract` hand-off using the lag-aware toolkit.

The five steps are:

1. **Triage** — `run_triage` on the target series (readiness, seasonality, primary AR lags).
2. **Select** — `run_lag_aware_mod_mrmr` on the covariates (legality + redundancy filtering).
3. **Contract** — `build_forecast_prep_contract` with `lag_aware_result=...`.
4. **Inspect** — render the contract markdown and compact lag table.
5. **Sanity check** — pass the contract's feature set to MLForecast + LightGBM
   on the CausalRivers dataset to confirm the selected features produce sensible skill.

Steps 1–4 are framework-agnostic. Step 5 is illustrative only.

## Setup

The cell below imports the public API, resolves the repo root, switches Matplotlib to `Agg`,
and pre-creates the output directories under `outputs/notebooks/recipes/`.

> **Prerequisite:** these notebooks require the local core checkout until v0.4.3 is published.
>
> ```bash
> export FORECASTABILITY_LOCAL_DEV=1
> bash ../ami/scripts/bootstrap_local_workspace.sh
> ```

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

try:
    from forecastability import (
        LagAwareModMRMRConfig,
        PairwiseScorerSpec,
        TriageRequest,
        build_forecast_prep_contract,
        forecast_prep_contract_to_lag_table,
        forecast_prep_contract_to_markdown,
        run_lag_aware_mod_mrmr,
        run_triage,
    )
except ImportError as exc:
    raise ImportError(
        "Lag-aware public imports are unavailable in the installed forecastability package. "
        "These notebooks require the local core checkout until v0.4.3 is published.\n\n"
        "Run:\n"
        "export FORECASTABILITY_LOCAL_DEV=1\n"
        "bash ../ami/scripts/bootstrap_local_workspace.sh"
    ) from exc

from forecastability.triage import AnalysisGoal

_HERE = Path().resolve()
if (_HERE / "pyproject.toml").exists() and (_HERE / "recipes").exists():
    REPO_ROOT = _HERE
elif _HERE.name == "recipes" and (_HERE.parent / "pyproject.toml").exists():
    REPO_ROOT = _HERE.parent
else:
    raise RuntimeError(
        "Run this notebook from the forecastability-examples repo root or its recipes/ directory."
    )

NOTEBOOK_STEM = 'lag_aware_mod_mrmr_to_forecast_prep_contract'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'notebooks' / 'recipes' / NOTEBOOK_STEM
FIG_DIR = OUTPUT_ROOT / 'figures'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.precision', 4)


def _rel(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


print(f"Figure dir: {_rel(FIG_DIR)}")
print(f"Table dir : {_rel(TABLE_DIR)}")

Figure dir: outputs/notebooks/recipes/lag_aware_mod_mrmr_to_forecast_prep_contract/figures
Table dir : outputs/notebooks/recipes/lag_aware_mod_mrmr_to_forecast_prep_contract/tables


## Step 1 — Build a Small Panel

The panel has four covariates:

| Column | Role |
|---|---|
| `promo_index` | True driver — latent AR(1), enters target at lag 3 |
| `promo_duplicate` | Near-duplicate of `promo_index` — should be suppressed |
| `calendar_flag` | Known-future — binary weekend flag, declared via `known_future_covariates` |
| `noise_sensor` | Pure noise — no signal |

The target depends on `promo_index` at lag 3 and `calendar_flag` contemporaneously.

In [2]:
def _ar1(n: int, *, phi: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    values = np.zeros(n, dtype=float)
    noise = rng.normal(scale=1.0, size=n)
    for idx in range(1, n):
        values[idx] = phi * values[idx - 1] + noise[idx]
    return values


rng = np.random.default_rng(SEED)
n = 480
time_index = np.arange(n, dtype=int)
latent_driver = _ar1(n, phi=0.82, seed=SEED + 1)
calendar_flag = ((time_index % 7) >= 5).astype(float)

target = np.zeros(n, dtype=float)
noise = rng.normal(scale=0.16, size=n)
for idx in range(5, n):
    target[idx] = (
        0.60 * target[idx - 1]
        - 0.12 * target[idx - 4]
        + 0.38 * latent_driver[idx - 3]
        + 0.10 * calendar_flag[idx]
        + noise[idx]
    )

panel = pd.DataFrame(
    {
        'target': target,
        'promo_index': latent_driver + 0.08 * rng.normal(size=n),
        'promo_duplicate': latent_driver + 0.02 * rng.normal(size=n),
        'calendar_flag': calendar_flag,
        'noise_sensor': rng.normal(size=n),
    },
    index=pd.RangeIndex(n, name='t'),
)

covariates = {
    column: panel[column].to_numpy(dtype=float)
    for column in panel.columns
    if column != 'target'
}

display(panel.head())

,target,promo_index,promo_duplicate,calendar_flag,noise_sensor
t,,,,,
0,0.0,0.0178,0.0323,0.0,-0.6646
1,0.0,0.7928,0.6808,0.0,0.3606
2,0.0,-0.0221,-0.0495,0.0,-1.3907
3,0.0,-0.8863,-0.9350,0.0,1.4387
4,0.0,-2.7613,-2.7574,0.0,-0.1440


## Step 2 — Triage the Target and Select Covariates

`run_triage` identifies the target's AR structure: readiness, seasonality, and primary lags.
The primary lags inform the `target_lags` argument — these are the self-lag indices the
forecast model will use from the target's own history, which ModMRMR uses to compute
target-history novelty.

`run_lag_aware_mod_mrmr` then scores all `(covariate, lag)` candidates via:

$$
\text{score}(j) = \text{relevance}(j) \cdot \bigl(1 - \max_{i \in S}\,\text{redundancy}(i,j)\bigr) \cdot \bigl(1 - \text{target\_history\_redundancy}(j)\bigr)
$$

Candidates with `lag < forecast_horizon + availability_margin` are **blocked before scoring** — they
would require future information at prediction time (default `availability_margin=1`).

In [3]:
triage_result = run_triage(
    TriageRequest(
        series=panel['target'].to_numpy(dtype=float),
        goal=AnalysisGoal.univariate,
        max_lag=16,
        n_surrogates=99,
        random_state=SEED,
    )
)

relevance_spec = PairwiseScorerSpec(
    name='mutual_info_sklearn',
    backend='sklearn',
    normalization='rank_percentile',
    significance_method='none',
    settings={'n_neighbors': 8},
)
similarity_spec = PairwiseScorerSpec(
    name='spearman_abs',
    backend='scipy',
    normalization='rank_percentile',
    significance_method='none',
)

lag_aware_config = LagAwareModMRMRConfig(
    forecast_horizon=2,
    availability_margin=0,
    candidate_lags=[1, 2, 3, 4, 5],
    known_future_covariates={'calendar_flag': 'calendar'},
    target_lags=[2, 4],
    max_selected_features=4,
    relevance_scorer=relevance_spec,
    redundancy_scorer=similarity_spec,
    target_history_scorer=similarity_spec,
)

lag_aware_result = run_lag_aware_mod_mrmr(
    target=panel['target'].to_numpy(dtype=float),
    covariates=covariates,
    config=lag_aware_config,
    random_state=SEED,
    run_id='recipe_lag_aware_to_contract',
)

selected_df = pd.DataFrame(
    [
        {
            'feature_name': item.feature_name,
            'covariate_name': item.covariate_name,
            'lag': item.lag,
            'known_future': item.is_known_future,
            'known_future_provenance': item.known_future_provenance,
            'selection_rank': item.selection_rank,
            'relevance': item.relevance,
            'max_redundancy': item.max_redundancy,
            'target_history_redundancy': item.target_history_redundancy,
            'final_score': item.final_score,
        }
        for item in lag_aware_result.selected
    ]
)

print(f'Triage readiness: {triage_result.readiness.status}')
print(f'Triage primary lags: {triage_result.interpretation.primary_lags if triage_result.interpretation is not None else []}')
display(selected_df)

Triage readiness: clear
Triage primary lags: [1]


,feature_name,covariate_name,lag,known_future,known_future_provenance,selection_rank,relevance,max_redundancy,target_history_redundancy,final_score
0,x_noise_sensor_lag3,noise_sensor,3,False,NaN,1,0.5294,0.0000,0.3235,0.3581
1,x_calendar_flag_lag4,calendar_flag,4,True,calendar,2,0.2941,0.1103,0.1471,0.2232
2,x_promo_index_lag3,promo_index,3,False,NaN,3,0.9412,0.3824,0.7059,0.1710
3,x_noise_sensor_lag2,noise_sensor,2,False,NaN,4,0.4118,0.8162,0.2941,0.0534


## Step 3 — Build the Contract and Inspect Covariate Roles

`build_forecast_prep_contract` wraps triage + lag-aware results into a single hand-off
object. The rendered markdown and compact lag rows make the past/future covariate split
immediately auditable before any downstream model is wired up.

In [4]:
contract = build_forecast_prep_contract(
    triage_result,
    horizon=2,
    target_frequency='D',
    lag_aware_result=lag_aware_result,
    add_calendar_features=False,
)

contract_markdown = forecast_prep_contract_to_markdown(contract)
contract_rows_df = pd.DataFrame(forecast_prep_contract_to_lag_table(contract))
role_df = pd.DataFrame(
    [
        {'bucket': 'past_covariates', 'names': contract.past_covariates},
        {'bucket': 'future_covariates', 'names': contract.future_covariates},
    ]
)

contract_markdown_path = TABLE_DIR / 'forecast_prep_contract.md'
contract_rows_path = TABLE_DIR / 'forecast_prep_contract_rows.csv'
role_path = TABLE_DIR / 'forecast_prep_contract_roles.csv'
contract_markdown_path.write_text(contract_markdown, encoding='utf-8')
contract_rows_df.to_csv(contract_rows_path, index=False)
role_df.to_csv(role_path, index=False)

print(f'Wrote contract markdown: {_rel(contract_markdown_path)}')
print(f'Wrote contract rows: {_rel(contract_rows_path)}')
print(f'Wrote role summary: {_rel(role_path)}')

display(Markdown(contract_markdown))
display(contract_rows_df)
display(role_df)

Wrote contract markdown: outputs/notebooks/recipes/lag_aware_mod_mrmr_to_forecast_prep_contract/tables/forecast_prep_contract.md
Wrote contract rows: outputs/notebooks/recipes/lag_aware_mod_mrmr_to_forecast_prep_contract/tables/forecast_prep_contract_rows.csv
Wrote role summary: outputs/notebooks/recipes/lag_aware_mod_mrmr_to_forecast_prep_contract/tables/forecast_prep_contract_roles.csv


# Forecast Prep Contract

## Metadata

- source_goal: lagged_exogenous
- blocked: False
- readiness_status: clear
- confidence_label: medium
- target_frequency: D
- horizon: 2
- contract_version: 0.3.4

## Target Lags

**recommended_target_lags:**
- 1

**recommended_seasonal_lags:**
(none)

**excluded_target_lags:**
(none)

**lag_rationale:**
- lag 1 is the strongest non-seasonal lag

## Model Families

**recommended_families:**
(none)

**baseline_families:**
- naive
- seasonal_naive

## Covariates

**past_covariates:**
- noise_sensor
- promo_index

**selected_covariate_lags:**
| axis | kind | driver | selected_lags | feature_names |
| --- | --- | --- | --- | --- |
| future | known_future:calendar | calendar_flag | 4 | x_calendar_flag_lag4 |
| past | measured | noise_sensor | 2, 3 | x_noise_sensor_lag2, x_noise_sensor_lag3 |
| past | measured | promo_index | 3 | x_promo_index_lag3 |

**covariate_notes:**
- future covariate calendar_flag: lags [4]
- past covariate noise_sensor: lags [2, 3]
- past covariate promo_index: lags [3]

**future_covariates:**
- calendar_flag

**calendar_features:**
(none)

**calendar_locale:** None

**rejected_covariates:**
(none)

**target_history_context:**
- enabled: True
- target_lags: [2, 4]
- scorer_name: spearman_abs
- normalization_strategy: rank_percentile
- penalized_selected_features: 4
- max_selected_redundancy: 0.7058823529411765
- notes:
- target-history novelty scored with spearman_abs over target lags [2, 4]
- 4 selected feature(s) carried non-zero target-history redundancy

## Notes

**caution_flags:**
(none)

**downstream_notes:**
(none)

**transformation_hints:**
(none)


,driver,axis,role,lag,selected_for_handoff,rationale,feature_name,future_known_required,known_future_provenance,kind
0,target,target,direct,1,True,lag 1 is the strongest non-seasonal lag,NaN,NaN,NaN,NaN
1,noise_sensor,past,past,2,True,"lag-aware ModMRMR selected sparse measured lags: [2, 3]",x_noise_sensor_lag2,False,NaN,measured
2,noise_sensor,past,past,3,True,"lag-aware ModMRMR selected sparse measured lags: [2, 3]",x_noise_sensor_lag3,False,NaN,measured
3,promo_index,past,past,3,True,lag-aware ModMRMR selected sparse measured lags: [3],x_promo_index_lag3,False,NaN,measured
4,calendar_flag,future,future,4,True,lag-aware ModMRMR selected known-future lags: [4] (provenance=calendar),x_calendar_flag_lag4,True,calendar,known_future:calendar


,bucket,names
0,past_covariates,"[noise_sensor, promo_index]"
1,future_covariates,[calendar_flag]


## Step 4 — MLForecast Sanity Check (CausalRivers)

This cell is self-contained: it loads the **CausalRivers** 8-station panel, runs
triage + Catt-scored ModMRMR, builds a contract, and then runs the same four-panel
deep dive as notebook 07:

| Panel | What it shows |
|---|---|
| (a) Forecast vs context | Single-origin h=4 prediction vs observed + naive |
| (b) Per-horizon MAE | How error grows with horizon (frozen-model walk-forward) |
| (c) Actual vs predicted | Scatter for all walk-forward windows |
| (d) Feature importance | Gain importance coloured by driver role (green=positive, red=negative) |

> **Evaluation note:** The model is fitted once on the full training set. The walk-forward
> origins overlap that training set — treat the walk-forward metrics as a frozen-model
> consistency check, not a held-out benchmark. For a rigorous evaluation, retrain at each origin.

> This is a hand-off sanity check, not a replacement for proper out-of-sample benchmarking.

In [ ]:
# --- Step 4: MLForecast deep dive on CausalRivers ---
# This cell is self-contained: it loads CausalRivers, runs triage + ModMRMR,
# builds the contract, then runs the four-panel deep dive.
# Requires: uv sync --extra mlforecast
#
# NOTE: model is fitted once on df_train. The walk-forward origins are within
# the training set — treat the walk-forward metrics as a frozen-model consistency
# check, not a held-out benchmark.

try:
    import subprocess
    import numpy as np
    import pandas as pd
    import lightgbm as lgb
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from IPython.display import Image, display
    from mlforecast import MLForecast
    from forecastability import (
        AnalysisGoal,
        LagAwareModMRMRConfig,
        PairwiseScorerSpec,
        TriageRequest,
        build_forecast_prep_contract,
        run_lag_aware_mod_mrmr,
        run_triage,
    )

    def _p(*args, **kw):
        kw["flush"] = True
        print(*args, **kw)

    # ── 4a. Load CausalRivers ─────────────────────────────────────────────
    _CR_TARGET_ID   = 978
    _CR_POSITIVES   = [979, 1095, 313, 758, 490]
    _CR_NEGATIVES   = [67, 71, 99]
    _CR_ALL_DRIVERS = _CR_POSITIVES + _CR_NEGATIVES
    _CR_HORIZON     = 4

    _data_path = REPO_ROOT / "data" / "causal_rivers" / "east_germany_8stations_6h.parquet"
    if not _data_path.exists():
        subprocess.run(
            [__import__("sys").executable, str(REPO_ROOT / "scripts" / "download_causal_rivers.py")],
            cwd=str(REPO_ROOT), check=True,
        )
    _frame = __import__("pandas").read_parquet(_data_path)
    _frame.columns = [int(c) for c in _frame.columns]
    _cr_panel = _frame[[_CR_TARGET_ID, *_CR_ALL_DRIVERS]].dropna()
    _cr_target = _cr_panel[_CR_TARGET_ID].to_numpy(dtype=float)
    _cr_covariates = {
        f"station_{sid}": _cr_panel[sid].to_numpy(dtype=float)
        for sid in _CR_ALL_DRIVERS
    }
    _p(f"CausalRivers panel: {_cr_panel.shape[0]} rows × {_cr_panel.shape[1]} stations")

    # ── 4b. Triage ────────────────────────────────────────────────────────
    _cr_triage = run_triage(
        TriageRequest(
            series=_cr_target,
            goal=AnalysisGoal.univariate,
            max_lag=24,
            n_surrogates=99,
            random_state=SEED,
        )
    )
    _raw_lags = list(_cr_triage.interpretation.primary_lags) if _cr_triage.interpretation else []
    _cr_target_lags = [lg for lg in _raw_lags if lg >= _CR_HORIZON]
    if len(_cr_target_lags) < 2:
        _cr_target_lags = [4, 6, 8]
    else:
        _cr_target_lags = _cr_target_lags[:4]
    _p(f"Target self-lags: {_cr_target_lags}")

    # ── 4c. Run lag-aware ModMRMR (Catt / KSG mode) ───────────────────────
    _catt_spec = PairwiseScorerSpec(name="catt_knn_mi")
    _catt_config = LagAwareModMRMRConfig(
        forecast_horizon=_CR_HORIZON,
        candidate_lags=list(range(1, 13)),
        target_lags=_cr_target_lags,
        max_selected_features=12,
        relevance_scorer=_catt_spec,
        redundancy_scorer=_catt_spec,
        target_history_scorer=_catt_spec,
        random_state=SEED,
    )
    _cr_result = run_lag_aware_mod_mrmr(
        target=_cr_target,
        covariates=_cr_covariates,
        config=_catt_config,
    )
    _cr_contract = build_forecast_prep_contract(
        _cr_triage,
        horizon=_CR_HORIZON,
        target_frequency="6h",
        lag_aware_result=_cr_result,
        add_calendar_features=False,
    )
    exog_cols = sorted(_cr_contract.past_covariates)
    _p(f"Selected covariates: {exog_cols}")

    # ── 4d. Build feature DataFrame ───────────────────────────────────────
    per_driver_lags: dict[str, list[int]] = {}
    for feature in _cr_result.selected:
        if feature.covariate_name in exog_cols:
            per_driver_lags.setdefault(feature.covariate_name, []).append(feature.lag)

    target_self_lags = _cr_target_lags
    max_target_lag   = max(target_self_lags)
    history_needed   = max_target_lag + _CR_HORIZON + 10

    _p("Building feature matrix ...", end=" ")
    station_ids = [int(s.split("_")[1]) for s in exog_cols]
    df_full = _cr_panel[[_CR_TARGET_ID, *station_ids]].copy()
    df_full.columns = ["y", *exog_cols]
    df_full["unique_id"] = "unstrut_978"
    df_full["ds"] = _cr_panel.index
    df_full = df_full.reset_index(drop=True)[["unique_id", "ds", "y", *exog_cols]]

    for col in exog_cols:
        for lag in per_driver_lags.get(col, []):
            df_full[f"{col}_lag{lag}"] = df_full[col].shift(lag)

    df_full       = df_full.drop(columns=exog_cols).dropna().reset_index(drop=True)
    lag_feat_cols = [c for c in df_full.columns if c not in ("unique_id", "ds", "y")]
    _p(f"done  ({len(df_full)} rows, {len(lag_feat_cols)} exog lag features)")

    # ── 4e. Fit model ─────────────────────────────────────────────────────
    horizon           = _CR_HORIZON
    forecast_origin   = len(df_full) - horizon
    forecast_start_ts = df_full["ds"].iloc[forecast_origin]
    df_train    = df_full.iloc[:forecast_origin].copy()
    window_test = df_full.iloc[forecast_origin : forecast_origin + horizon].copy()
    history     = df_full.iloc[max(0, forecast_origin - history_needed):forecast_origin].copy()
    X_future    = window_test[["unique_id", "ds"] + lag_feat_cols]

    # n_jobs=1: avoids macOS OpenMP/fork deadlock in notebook kernels
    model = lgb.LGBMRegressor(
        n_estimators=100, learning_rate=0.07, num_leaves=31,
        random_state=SEED, verbosity=-1, n_jobs=1,
    )
    mlf = MLForecast(models={"lgbm": model}, freq="6h", lags=target_self_lags)

    _p(f"\n[1/4] Fitting LightGBM on {len(df_train)} train rows ...", end=" ")
    mlf.fit(df_train, static_features=[], id_col="unique_id", time_col="ds", target_col="y")
    _p("done")

    # ── 4f. Single-origin forecast ────────────────────────────────────────
    _p(f"[2/4] Single-origin forecast (origin={forecast_start_ts}) ...", end=" ")
    fc     = mlf.predict(h=horizon, new_df=history, X_df=X_future)
    actual = window_test["y"].to_numpy()
    pred   = fc["lgbm"].to_numpy()
    ds     = window_test["ds"].to_numpy()
    _p("done")

    mae        = float(np.mean(np.abs(actual - pred)))
    rmse       = float(np.sqrt(np.mean((actual - pred) ** 2)))
    ss         = float(np.sum((actual - np.mean(actual)) ** 2))
    nse        = float(1.0 - np.sum((actual - pred) ** 2) / ss) if ss > 1e-12 else float("nan")
    last_known = float(df_full["y"].iloc[forecast_origin - 1])
    naive_pred = np.full(horizon, last_known)
    naive_mae  = float(np.mean(np.abs(actual - naive_pred)))
    _p(f"  LGBM  : MAE={mae:.3f}  RMSE={rmse:.3f}  NSE={nse:.4f}")
    _p(f"  Naive : MAE={naive_mae:.3f}")

    # ── 4g. Walk-forward (frozen model) ───────────────────────────────────
    _p(f"\n[3/4] Walk-forward evaluation (10 origins × h={horizon}) ...")
    wf_results = []
    wf_origins = list(range(len(df_full) - 10 * horizon, len(df_full) - horizon, horizon))
    for i, orig in enumerate(wf_origins):
        _p(f"  origin {i + 1}/{len(wf_origins)} (row {orig}) ...", end=" ")
        _hist = df_full.iloc[max(0, orig - history_needed):orig].copy()
        _win  = df_full.iloc[orig : orig + horizon].copy()
        _Xf   = _win[["unique_id", "ds"] + lag_feat_cols]
        try:
            _fc   = mlf.predict(h=horizon, new_df=_hist, X_df=_Xf)
            _act  = _win["y"].to_numpy()
            _prd  = _fc["lgbm"].to_numpy()
            _last = float(df_full["y"].iloc[orig - 1])
            for h_i in range(horizon):
                wf_results.append({
                    "h": h_i + 1,
                    "actual": _act[h_i], "pred": _prd[h_i], "naive": _last,
                    "err": _act[h_i] - _prd[h_i],
                    "naive_err": _act[h_i] - _last,
                })
            _p(f"MAE={float(np.mean(np.abs(_act - _prd))):.3f}")
        except Exception as _ex:
            _p(f"SKIP ({_ex})")

    wf     = pd.DataFrame(wf_results)
    per_h  = wf.groupby("h").apply(
        lambda g: pd.Series({"MAE": g["err"].abs().mean(), "Naive_MAE": g["naive_err"].abs().mean()})
    )
    wf_mae   = float(wf["err"].abs().mean())
    wf_naive = float(wf["naive_err"].abs().mean())
    skill    = (1 - wf_mae / wf_naive) * 100
    _p(f"  Overall walk-forward  MAE={wf_mae:.3f}  Naive={wf_naive:.3f}  skill={skill:.1f}%")

    # ── 4h. Feature importance + 4-panel plot ─────────────────────────────
    _p("\n[4/4] Plotting ...", end=" ")
    fitted_model = mlf.models_["lgbm"]
    feat_names   = mlf.ts.features_order_
    fi = pd.Series(fitted_model.feature_importances_, index=feat_names).sort_values(ascending=False)

    _POS_IDS = set(_CR_POSITIVES)

    def _feat_color(name: str) -> str:
        if name.startswith("lag"):
            return "tab:blue"
        for sid in _POS_IDS:
            if name.startswith(f"station_{sid}_"):
                return "tab:green"
        return "tab:red"

    context_steps = 8 * horizon
    plot_start    = max(0, forecast_origin - context_steps)
    context_df    = df_full.iloc[plot_start:forecast_origin]

    fig = plt.figure(figsize=(16, 13))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.35)

    ax0 = fig.add_subplot(gs[0, :])
    ax0.plot(context_df["ds"], context_df["y"], lw=1.0, color="tab:blue", label="Observed (context)")
    ax0.plot(ds, actual, lw=2.5, marker="o", color="black",     label="Observed (test)", zorder=5)
    ax0.plot(ds, pred,   lw=2.5, marker="o", color="tab:green", label=f"LGBM (MAE={mae:.2f}, NSE={nse:.3f})", zorder=6)
    ax0.plot(ds, naive_pred, lw=1.5, ls="--", color="tab:red", label=f"Naive (MAE={naive_mae:.2f})", zorder=4)
    ax0.axvline(forecast_start_ts, color="gray", ls=":", lw=1.5)
    ax0.set_ylabel("Flow")
    ax0.legend(fontsize=8, loc="upper left")
    ax0.grid(alpha=0.25)
    ax0.set_title(f"Single-origin h={horizon} forecast — target lags={target_self_lags}")
    plt.setp(ax0.xaxis.get_majorticklabels(), rotation=20, ha="right")

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.bar(per_h.index - 0.2, per_h["MAE"],       0.38, label="LGBM",  color="tab:green", alpha=0.85)
    ax1.bar(per_h.index + 0.2, per_h["Naive_MAE"], 0.38, label="Naive", color="tab:red",   alpha=0.6)
    ax1.set_xlabel("Horizon step")
    ax1.set_ylabel("MAE")
    ax1.set_title(f"Per-horizon MAE (walk-forward, {len(wf_origins)} origins)")
    ax1.legend()
    ax1.grid(alpha=0.25, axis="y")

    ax2 = fig.add_subplot(gs[1, 1])
    ax2.scatter(wf["actual"], wf["pred"], alpha=0.35, s=10, color="tab:green", label="LGBM")
    _lim = [wf["actual"].min(), wf["actual"].max()]
    ax2.plot(_lim, _lim, "k--", lw=1, label="1:1")
    ax2.set_xlabel("Actual flow")
    ax2.set_ylabel("Predicted flow")
    ax2.set_title("Walk-forward: actual vs predicted")
    ax2.legend()
    ax2.grid(alpha=0.25)

    ax3 = fig.add_subplot(gs[2, :])
    top  = fi.head(20)
    cols = [_feat_color(n) for n in top.index]
    ax3.barh(top.index[::-1], top.values[::-1], color=cols[::-1])
    ax3.set_xlabel("LightGBM feature importance (gain)")
    ax3.set_title("Feature importance  |  blue=target AR lag  green=positive driver  red=negative/control driver")
    ax3.grid(alpha=0.25, axis="x")

    plot_path = FIG_DIR / "mlforecast_causal_rivers_deep_dive.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    _p(f"done\nSaved: {_rel(plot_path)}")

    _p("\nTop-10 features:")
    for fname, imp in fi.head(10).items():
        color_tag = {"tab:blue": "[AR]", "tab:green": "[+]", "tab:red": "[-]"}[_feat_color(fname)]
        _p(f"  {color_tag}  {fname:<40} {imp:.0f}")

    display(Image(filename=str(plot_path)))

except ImportError as _ie:
    print(f"mlforecast/lightgbm not installed — skipping. ({_ie})", flush=True)
except Exception as _e:
    import traceback
    print(f"MLForecast error: {type(_e).__name__}: {_e}", flush=True)
    traceback.print_exc()


## Summary

The five-step path shown in this recipe:

```
run_triage(...)          → readiness, primary AR lags
    ↓
run_lag_aware_mod_mrmr(...)  → selected (covariate, lag) pairs
    ↓
build_forecast_prep_contract(...)  → framework-agnostic hand-off
    ↓
forecast_prep_contract_to_markdown / to_lag_table  → human-readable summary
    ↓
MLForecast / any downstream library  → model fitting (outside this toolkit)
```

The compact lag table is the easiest surface to audit before any framework wiring:
confirm that past and future covariates are separated as intended, that blocked lags
are absent, and that known-future provenance is correctly labelled.